In [1]:
import sys
import os
import datasets
from datasets import Dataset, concatenate_datasets
from typing import List, Dict, Any
import random
import json

from transformers import AutoTokenizer

# Add the project root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [2]:
data_path = "../../dataset/train.json"
dataset = datasets.load_dataset("json", data_files=data_path)
original_train_list = dataset["train"].to_list()

context_data_path = "../../dataset/processed_data/train/rajpurkar_squad_processed.json"
context_dataset = datasets.load_dataset("json", data_files=context_data_path)

math_data_path = "../../dataset/processed_data/train/UMWP_processed.json"
math_dataset = datasets.load_dataset("json", data_files=math_data_path)

In [3]:
print(dataset)
print(context_dataset)
print(math_dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'incorrect_response', 'errors', 'hallucinated_text', 'correct_response', 'additional_info'],
        num_rows: 40759
    })
})
DatasetDict({
    train: Dataset({
        features: ['input', 'question', 'context', 'answer', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
        num_rows: 87557
    })
})
DatasetDict({
    train: Dataset({
        features: ['input', 'question', 'answer', 'is_answerable', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
        num_rows: 4413
    })
})


### Find math qa data samples that does not have any errors

In [4]:
# find data samples for math dataset with no errors 
math_dataset = math_dataset.filter(lambda x: x["wrong_response_number"] == 0)
print(math_dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'question', 'answer', 'is_answerable', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
        num_rows: 1575
    })
})


In [5]:
def format_single_sample(sample: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Formats a single raw sample into one or more training samples."""

    sample_kwargs = {
        "question": sample["question"],
        "answer": str(sample.get("answer", [""])[0]),
    }
    if "is_answerable" in sample:
        sample_kwargs["is_answerable"] = sample["is_answerable"]
    else:
        sample_kwargs["context"] = sample.get("context", "")
    
    formatted_samples = []
    responses = sample["responses"]
    # sort responses by length
    responses.sort(key=len)
    samples_to_include = 1 if random.random() < 0.8 else 2


    for i in range(samples_to_include):
        formatted_samples.append({
            "input": sample["input"],
            "incorrect_response": "",
            "errors": [],
            "hallucinated_text": [],
            "correct_response": responses[i],
            "additional_info": sample_kwargs,
        })

    return formatted_samples

In [6]:
final_math_dataset = []
for sample in math_dataset["train"]:
    final_math_dataset.extend(format_single_sample(sample))

random.shuffle(final_math_dataset)
print(len(final_math_dataset))

1904


In [7]:
final_math_dataset[0]

{'input': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a meticulous AI mathematician. Your task is to solve the following math problem.\n\nFollow these steps carefully:\n1. **Analyze the problem:** First, understand the given information and what is being asked.\n2. **Assess solvability:** Determine if the problem is solvable. A problem might be unsolvable if it's illogical, contains contradictions, or lacks necessary information.\n3. **Solve or Explain:**\n   - **If solvable:** Provide a step-by-step solution, showing all your reasoning and calculations, and then clearly state the final numerical answer.\n   - **If unsolvable:** State that the problem cannot be answered and provide a concise explanation.\n\nYour entire response should only contain the solution and final answer (or the explanation for unsolvable problems). Do not add any conversational headers or extraneous text.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nA construction company is b

In [8]:
original_train_list.extend(final_math_dataset)
random.shuffle(original_train_list)
print(original_train_list[0])
print(len(original_train_list))

{'input': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a specialized question-answering AI. Your task is to give a concise answer to the question using *only* the provided context. Make sure to always give an answer.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nContext:\n\'\'\'\nJerusalem with its many holy places probably had the highest concentration of mosaic-covered churches but very few of them survived the subsequent waves of destructions. The present remains do not do justice to the original richness of the city. The most important is the so-called "Armenian Mosaic" which was discovered in 1894 on the Street of the Prophets near Damascus Gate. It depicts a vine with many branches and grape clusters, which springs from a vase. Populating the vine\'s branches are peacocks, ducks, storks, pigeons, an eagle, a partridge, and a parrot in a cage. The inscription reads: "For the memory and salvation of all those Armenians whose name the Lord knows." B

### Find Context QA that does not have any errors

In [9]:
# find data samples for context dataset with no errors 
context_dataset = context_dataset.filter(lambda x: x["wrong_response_number"] == 0)
print(context_dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'question', 'context', 'answer', 'responses', 'errors', 'wrong_response_number', 'responses_to_correct', 'errors_to_correct', 'corrected_responses', 'verified_response_mask'],
        num_rows: 70203
    })
})


In [10]:
def format_single_sample(sample: Dict[str, Any]) -> List[Dict[str, Any]]:
    """Formats a single raw sample into one or more training samples."""

    sample_kwargs = {
        "question": sample["question"],
        "answer": str(sample.get("answer", [""])[0]),
    }
    if "is_answerable" in sample:
        sample_kwargs["is_answerable"] = sample["is_answerable"]
    else:
        sample_kwargs["context"] = sample.get("context", "")
    
    formatted_samples = []
    responses = sample["responses"]
    # sort responses by length
    responses.sort(key=len)
    samples_to_include = 1

    for i in range(samples_to_include):
        formatted_samples.append({
            "input": sample["input"],
            "incorrect_response": "",
            "errors": [],
            "hallucinated_text": [],
            "correct_response": responses[i],
            "additional_info": sample_kwargs,
        })

    return formatted_samples

In [11]:
final_context_dataset = []
samples_to_include = 9000
for i in range(samples_to_include):
    final_context_dataset.extend(format_single_sample(context_dataset["train"][i]))

random.shuffle(final_context_dataset)
print(len(final_context_dataset))

9000


In [12]:
original_train_list.extend(final_context_dataset)
random.shuffle(original_train_list)
print(original_train_list[0])
print(len(original_train_list))

{'input': "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nYou are a specialized question-answering AI. Your task is to give a concise answer to the question using *only* the provided context. Make sure to always give an answer.<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nContext:\n'''\nStrasbourg (/ˈstræzbɜːrɡ/, French pronunciation: \u200b[stʁaz.buʁ, stʁas.buʁ]; Alsatian: Strossburi; German: Straßburg, [ˈʃtʁaːsbʊɐ̯k]) is the capital and largest city of the Alsace-Champagne-Ardenne-Lorraine (ACAL) region in eastern France and is the official seat of the European Parliament. Located close to the border with Germany, it is the capital of the Bas-Rhin département. The city and the region of Alsace were historically predominantly Alemannic-speaking, hence the city's Germanic name. In 2013, the city proper had 275,718 inhabitants, Eurométropole de Strasbourg (Greater Strasbourg) had 475,934 inhabitants and the Arrondissement of Strasbourg had 482,384 inhabitants. Wi

In [13]:
output_path = os.path.join(project_root, "full_train_dataset.json")
with open(output_path, "w") as f:
    json.dump(original_train_list, f, indent=4)

In [5]:
import datasets
from datasets import Dataset, DatasetDict
from huggingface_hub import HfFolder

import sys
import os
# Add the project root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [6]:
data_path = "../../dataset/full_train_dataset.json"
dataset = datasets.load_dataset("json", data_files=data_path)
dataset = dataset["train"].to_list()

In [7]:
len(dataset)

51663

In [ ]:
HUGGINGFACE_TOKEN_ENV_VAR = ""
repo_name = "MathBite/llama_wsa_self-correction"

In [10]:
token = HUGGINGFACE_TOKEN_ENV_VAR
if not token:
    raise ValueError(f"Hugging Face token not found. Set the {HUGGINGFACE_TOKEN_ENV_VAR} environment variable.")

HfFolder.save_token(token)

hf_dataset = Dataset.from_list(dataset)
dataset_dict = DatasetDict({"train": hf_dataset})

dataset_dict.push_to_hub(repo_name, private=True)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/52 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/775 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/MathBite/llama_wsa_self-correction/commit/6bc6aa8b9e2866897aa9068c832e32b979e22805', commit_message='Upload dataset', commit_description='', oid='6bc6aa8b9e2866897aa9068c832e32b979e22805', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/MathBite/llama_wsa_self-correction', endpoint='https://huggingface.co', repo_type='dataset', repo_id='MathBite/llama_wsa_self-correction'), pr_revision=None, pr_num=None)